In [1]:
import torch
import time
import os
import torchvision.models as models
from torchvision.io import read_image
from torchvision import transforms
from torchvision.models import AlexNet_Weights, VGG16_Weights, ResNet50_Weights
from tabulate import tabulate

Ссылка к папке с изображениями

In [2]:
path = "images/"
images = [f for f in os.listdir(path) if f.endswith(('.png', '.jpg', '.JPEG'))]

Обработка изображения перед его передачей в модель

In [3]:
def preprocess_image(img):
    preprocess = transforms.Compose([
        transforms.Resize(256), # нормализация
        transforms.CenterCrop(224), # Обрезать фото для Vgg16,ResNet, AlexNet
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    return preprocess(img / 255.0)

Делать прогнозы для модели

In [4]:
def predict_top5(model, preprocessed_img, categories, result):
    start_time = time.time()
    prediction = model(preprocessed_img.unsqueeze(0)).squeeze(0).softmax(0) #
    top5_accuracy, top5_id = torch.topk(prediction, 5)
    end_time = time.time()

    print(f"{model_name}: ")
    for i in range(5):
        class_id = top5_id[i].item()
        score = top5_accuracy[i].item()
        print(f"  {i+1}. {categories[class_id]} ({class_id}): {score:.2%}")
    print(f"Running time: {end_time - start_time}")
    top1 = categories[top5_id[0].item()]
    return top1.lower() == result.lower(),

Модель и соответствующий вес

In [5]:
models_weights = {
    "resnet50": (models.resnet50, ResNet50_Weights.DEFAULT),
    "alexnet": (models.alexnet, AlexNet_Weights.DEFAULT),
    "vgg16": (models.vgg16, VGG16_Weights.DEFAULT),
}

Перевести модели в режим оценки

In [7]:
models = {}
for name, (model_func, weights) in models_weights.items():
    model = model_func(weights=weights) # инициализации модели c предварительным обученным весом
    models[name] = model.eval()

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\asgat/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:03<00:00, 32.1MB/s]


In [8]:
# массив для хранения правильных прогнозов
results = []
correct_counts = {model_name: 0 for model_name in models.keys()}

Обработка и классификация каждого изображения в списке изображений

In [9]:
for image in images:
    img = read_image(os.path.join(path, image))
    preprocessed_img = preprocess_image(img)
    categories = weights.meta["categories"]
    result = os.path.splitext(image)[0].strip()

    print(f"\nImage: {image}")
    row = [image]
    for model_name, model in models.items():
        is_correct = predict_top5(model, preprocessed_img, categories, result)
        if is_correct: correct_counts[model_name] += 1
        row.append("Correct" if is_correct else "Wrong")

    results.append(row)


Image: American chameleon.JPEG
resnet50: 
  1. American chameleon (40): 33.57%
  2. green lizard (46): 17.28%
  3. agama (42): 0.27%
  4. common iguana (39): 0.24%
  5. alligator lizard (44): 0.23%
Running time: 0.30795955657958984
alexnet: 
  1. American chameleon (40): 72.98%
  2. green lizard (46): 26.73%
  3. tree frog (31): 0.10%
  4. agama (42): 0.07%
  5. common iguana (39): 0.06%
Running time: 0.08812236785888672
vgg16: 
  1. American chameleon (40): 65.53%
  2. green lizard (46): 34.13%
  3. agama (42): 0.14%
  4. banded gecko (38): 0.08%
  5. common iguana (39): 0.06%
Running time: 0.4313013553619385

Image: American egret.JPEG
resnet50: 
  1. American egret (132): 72.53%
  2. crane bird (134): 6.51%
  3. little blue heron (131): 0.90%
  4. spoonbill (129): 0.62%
  5. lakeside (975): 0.61%
Running time: 0.12918472290039062
alexnet: 
  1. American egret (132): 54.22%
  2. grey whale (147): 7.16%
  3. pelican (144): 4.91%
  4. albatross (146): 4.89%
  5. killer whale (148): 4.

In [10]:
headers = ["Image File"] + [model_name.capitalize() for model_name in models.keys()]
print("\nResults Table:")
print(tabulate(results, headers=headers, tablefmt="grid"))


Results Table:
+-------------------------+------------+-----------+---------+
| Image File              | Resnet50   | Alexnet   | Vgg16   |
+=========================+============+===========+=========+
| American chameleon.JPEG | Correct    | Correct   | Correct |
+-------------------------+------------+-----------+---------+
| American egret.JPEG     | Correct    | Correct   | Correct |
+-------------------------+------------+-----------+---------+
| American lobster.JPEG   | Correct    | Correct   | Correct |
+-------------------------+------------+-----------+---------+
| bee eater.JPEG          | Correct    | Correct   | Correct |
+-------------------------+------------+-----------+---------+
| black swan.JPEG         | Correct    | Correct   | Correct |
+-------------------------+------------+-----------+---------+
| coucal.JPEG             | Correct    | Correct   | Correct |
+-------------------------+------------+-----------+---------+
| Dungeness crab.JPEG     | Correct    

Точность каждой модели

In [11]:
print("\nFinal Results:")
for model_name, correct_count in correct_counts.items():
    accuracy = (correct_count / len(images)) * 100
    print(f"{model_name.capitalize()} - Correct Top-1 Predictions: {correct_count}/{len(images)} ({accuracy:.2f}%)")


Final Results:
Resnet50 - Correct Top-1 Predictions: 29/29 (100.00%)
Alexnet - Correct Top-1 Predictions: 29/29 (100.00%)
Vgg16 - Correct Top-1 Predictions: 29/29 (100.00%)
